# 📖 Notebook 2: Leader ElectionIn many distributed systems, you need **exactly one server** to be the "leader" — the one that coordinates work, writes to a database, or makes decisions.But what if the leader crashes? You need a new leader, fast. And you need to make sure there's **never more than one leader at the same time** (called "split-brain").In this notebook, we'll see three approaches:| Approach | Method | Problem ||----------|--------|---------|| 🔴 Bad | Random selection | No failover, split-brain possible || 🟡 Better | Database-based election | Requires polling, no crash detection || 🟢 Best | ZooKeeper sequential ephemeral nodes | Instant failover, no split-brain |## Learning ObjectivesBy the end of this notebook, you'll understand:- Why leader election is hard in distributed systems- What "split-brain" means and why it's dangerous- How ZooKeeper's sequential ephemeral nodes solve the problem- How automatic failover works when a leader crashes

## 🛠️ SetupStart the ZooKeeper ensemble first:```bashcd deep-dives/zookeeperdocker-compose up -d```### Kernel SelectionSelect the `.venv` kernel in VS Code's kernel picker (top-right of notebook).If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import threadingimport timeimport random

---## 🔴 Bad: Random Leader Selection### The ProblemThe simplest "leader election" is to just pick one at random. Each server starts up and decides if it's the leader based on a coin flip or random number.This has obvious problems:- **Multiple leaders**: Two servers might both think they're the leader- **No failover**: If the leader crashes, nobody knows and nobody takes over- **No agreement**: Servers don't communicate, so they can't agree on who's in chargeThis is like a group project where everyone just decides on their own who's the team lead.

In [ ]:
class NaiveServer:    """    BAD: Each server randomly decides if it's the leader.    No coordination between servers at all.    """    def __init__(self, server_id, num_servers):        self.server_id = server_id        # Each server independently "elects" itself with probability 1/N        self.is_leader = random.random() < (1.0 / num_servers)    def __repr__(self):        role = "LEADER" if self.is_leader else "follower"        return f"Server-{self.server_id} [{role}]"# Simulate 5 servers starting upprint("5 servers start up and each randomly decides if it's the leader:")print("=" * 60)# Run the experiment 5 times to show the randomnessfor trial in range(5):    servers = [NaiveServer(i, 5) for i in range(5)]    leaders = [s for s in servers if s.is_leader]    print(f"\nTrial {trial + 1}: {[str(s) for s in servers]}")    if len(leaders) == 0:        print("  ❌ NO LEADER — nobody is coordinating the work!")    elif len(leaders) > 1:        print(f"  ❌ SPLIT BRAIN — {len(leaders)} servers think they're the leader!")        print("     This can cause data corruption (e.g., two servers writing conflicting data).")    else:        print(f"  ✅ Got lucky — exactly one leader: {leaders[0]}")

In [ ]:
# Even worse: simulate a leader crash with no failoverprint("Simulating leader crash with random election:")print("=" * 60)servers = [NaiveServer(i, 5) for i in range(5)]# Force server 0 to be leader for the demofor s in servers:    s.is_leader = Falseservers[0].is_leader = Trueprint(f"Initial state: {[str(s) for s in servers]}")print(f"Leader is Server-0")print("\n💥 Server-0 crashes!")servers.pop(0)leaders = [s for s in servers if s.is_leader]print(f"Remaining servers: {[str(s) for s in servers]}")print(f"Number of leaders: {len(leaders)}")print()print("❌ NO FAILOVER — the system has no leader and no way to elect a new one!")print("   All coordination work stops until someone manually intervenes.")

---## 🟡 Better: Database-Based Election### The IdeaUse a shared database table as a coordination point. Servers try to insert their ID into a "leader" row. Only one can succeed (unique constraint). Other servers poll the table periodically to check if the leader is still alive.We'll simulate this with a Python dictionary (in production, this would be a database like PostgreSQL).### Why It's Better- Servers coordinate through a shared store- Only one leader at a time (enforced by the database)### Why It's Still Not Great- **Polling required**: Followers must constantly check if the leader is still alive- **Slow failover**: Leader crash isn't detected until the heartbeat expires- **Database is a single point of failure**: If the DB goes down, no election can happen

In [ ]:
class DatabaseElection:    """    Simulates a database-based leader election.    In production, this would use a real database with row-level locking.    """    def __init__(self):        self._leader = None        self._heartbeat = 0        self._lock = threading.Lock()        self.heartbeat_timeout = 3  # seconds before leader is considered dead    def try_become_leader(self, server_id):        """Try to claim leadership. Returns True if successful."""        with self._lock:            now = time.time()            if self._leader is None or (now - self._heartbeat > self.heartbeat_timeout):                old_leader = self._leader                self._leader = server_id                self._heartbeat = now                if old_leader and old_leader != server_id:                    print(f"  Server-{server_id} took over from Server-{old_leader} (heartbeat expired)")                return True            return self._leader == server_id    def send_heartbeat(self, server_id):        """Leader sends a heartbeat to prove it's still alive."""        with self._lock:            if self._leader == server_id:                self._heartbeat = time.time()                return True            return False    def get_leader(self):        """Check who the current leader is."""        with self._lock:            if self._leader and (time.time() - self._heartbeat <= self.heartbeat_timeout):                return self._leader            return Noneclass DatabaseServer:    def __init__(self, server_id, election):        self.server_id = server_id        self.election = election        self.running = True        self.is_leader = False    def run(self):        """Main loop: try to become leader, or poll for leader status."""        while self.running:            if self.is_leader:                self.election.send_heartbeat(self.server_id)            else:                if self.election.try_become_leader(self.server_id):                    self.is_leader = True                    print(f"  Server-{self.server_id} became the leader!")            time.sleep(1)  # poll every second# Run the database electionprint("Database-Based Leader Election")print("=" * 60)election = DatabaseElection()db_servers = [DatabaseServer(i, election) for i in range(3)]threads = [threading.Thread(target=s.run, daemon=True) for s in db_servers]for t in threads:    t.start()time.sleep(2)leader = election.get_leader()print(f"\nCurrent leader: Server-{leader}")# Simulate leader crashprint(f"\n💥 Server-{leader} crashes!")for s in db_servers:    if s.server_id == leader:        s.running = False        s.is_leader = False        breakprint(f"Waiting for heartbeat timeout ({election.heartbeat_timeout}s)...")time.sleep(election.heartbeat_timeout + 2)new_leader = election.get_leader()print(f"\nNew leader: Server-{new_leader}")# Stop all serversfor s in db_servers:    s.running = Falsetime.sleep(1)if new_leader and new_leader != leader:    print("\n⚠️  Failover worked, but it was SLOW:")    print(f"   Had to wait {election.heartbeat_timeout}+ seconds for the heartbeat to expire.")    print("   During that time, the system had no active leader.")

### Why Database Election Is Not Ideal| Limitation | Why It Matters ||------------|----------------|| Polling required | Each server must constantly check the database || Slow failover | Must wait for heartbeat timeout (seconds to minutes) || DB is a SPOF | If the database fails, no election can happen || Wasteful | Constant polling creates unnecessary database load |

---## 🟢 Best: ZooKeeper Leader Election### The SolutionZooKeeper uses **sequential ephemeral nodes** for leader election. Here's how it works:1. Each server creates a **sequential ephemeral node** under `/election`2. ZooKeeper assigns each node an incrementing sequence number3. The server with the **lowest sequence number** is the leader4. If the leader crashes, its ephemeral node vanishes → the next server becomes leader```Before crash:                After leader crash:/election                    /election  ├── node-0001 (Server A)     ├── node-0002 (Server B) ← new leader!  ├── node-0002 (Server B)     └── node-0003 (Server C)  └── node-0003 (Server C)```### Why This Is Better| Feature | Database Election | ZooKeeper Election ||---------|-------------------|--------------------|| Crash detection | Polling (slow) | Instant (watches) || Failover speed | Seconds-minutes | Sub-second || Split-brain risk | Possible | Impossible (ZAB consensus) || Server load | High (constant polling) | Low (event-driven) |

In [ ]:
from kazoo.client import KazooClient# Connect to our 3-node ZooKeeper ensemblezk = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")zk.start()print("Connected to ZooKeeper!")

In [ ]:
class ZooKeeperServer:    """    BEST: Uses ZooKeeper sequential ephemeral nodes for leader election.    - Instant failover when leader crashes    - No polling — uses ZooKeeper watches    - No split-brain — ZooKeeper's consensus protocol prevents it    """    def __init__(self, server_id, zk_hosts):        self.server_id = server_id        self.zk = KazooClient(hosts=zk_hosts)        self.zk.start()        self.is_leader = False        self.node_path = None    def join_election(self):        """Create a sequential ephemeral node to join the election."""        self.zk.ensure_path("/demo/election")        # Create a sequential ephemeral node        # - EPHEMERAL: auto-deleted when this client disconnects        # - SEQUENCE: ZooKeeper appends an incrementing number        self.node_path = self.zk.create(            "/demo/election/candidate-",            value=f"server-{self.server_id}".encode(),            ephemeral=True,            sequence=True        )        print(f"  Server-{self.server_id} joined election as {self.node_path}")        self._check_leadership()    def _check_leadership(self):        """Check if this server has the lowest sequence number (= leader)."""        children = self.zk.get_children("/demo/election")        children.sort()        my_node = self.node_path.split("/")[-1]        if children[0] == my_node:            self.is_leader = True            print(f"  👑 Server-{self.server_id} is the LEADER (lowest sequence: {my_node})")        else:            self.is_leader = False            print(f"     Server-{self.server_id} is a follower (node: {my_node})")    def crash(self):        """Simulate a server crash by disconnecting from ZooKeeper."""        self.zk.stop()# Run the ZooKeeper electionprint("ZooKeeper Leader Election")print("=" * 60)zk_hosts = "localhost:2181,localhost:2182,localhost:2183"# Clean up from any previous runif zk.exists("/demo/election"):    zk.delete("/demo/election", recursive=True)# 3 servers join the electionzk_servers = []for i in range(3):    server = ZooKeeperServer(i, zk_hosts)    server.join_election()    zk_servers.append(server)

In [ ]:
# Show the current election stateprint("Current election nodes in ZooKeeper:")print("=" * 60)children = zk.get_children("/demo/election")for child in sorted(children):    data, stat = zk.get(f"/demo/election/{child}")    is_lowest = child == sorted(children)[0]    marker = "👑 LEADER" if is_lowest else "   follower"    print(f"  {marker}: /demo/election/{child} → {data.decode()}")

In [ ]:
# Now simulate the leader crashingleader_srv = next(s for s in zk_servers if s.is_leader)print(f"💥 Server-{leader_srv.server_id} (the leader) crashes!")leader_srv.crash()print()# Give ZooKeeper a moment to detect the disconnectiontime.sleep(3)# Check who's the new leaderprint("After crash — election nodes:")children = zk.get_children("/demo/election")for child in sorted(children):    data, stat = zk.get(f"/demo/election/{child}")    is_lowest = child == sorted(children)[0]    marker = "👑 NEW LEADER" if is_lowest else "   follower"    print(f"  {marker}: /demo/election/{child} → {data.decode()}")print()print("✅ Leader failover happened automatically!")print("   No polling. No manual intervention. The crashed server's ephemeral node")print("   was deleted, and the next server in line became the leader.")

### 🔍 Using kazoo's Built-In Election RecipeThe kazoo library includes a high-level `Election` recipe that handles all the edge cases for you:

In [ ]:
# Clean upfor s in zk_servers:    try:        s.zk.stop()    except Exception:        passif zk.exists("/demo/election"):    zk.delete("/demo/election", recursive=True)# Use kazoo's built-in Election recipeleader_events = []def on_leader(server_id):    """Called when this server becomes the leader."""    leader_events.append(server_id)    print(f"  👑 Server-{server_id} became the leader via kazoo Election recipe!")    # In a real app, you'd start doing leader work here    time.sleep(5)# Create election participantsclient1 = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")client1.start()election1 = client1.Election("/demo/auto-election", "server-1")client2 = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")client2.start()election2 = client2.Election("/demo/auto-election", "server-2")# Run elections in background threadst1 = threading.Thread(target=election1.run, args=(lambda: on_leader(1),), daemon=True)t2 = threading.Thread(target=election2.run, args=(lambda: on_leader(2),), daemon=True)print("Starting election with kazoo Election recipe:")t1.start()time.sleep(0.5)t2.start()time.sleep(2)# Crash the first leaderprint(f"\n💥 Crashing client1...")client1.stop()time.sleep(4)print(f"\nLeadership history: {leader_events}")print("✅ kazoo's Election recipe handles failover automatically!")# Cleanuptry:    client2.stop()except Exception:    pass

---## 📊 Summary: Bad → Better → Best| | 🔴 Random Selection | 🟡 Database Election | 🟢 ZooKeeper Election ||---|---|---|---|| **One leader guaranteed** | ❌ Multiple possible | ✅ Unique constraint | ✅ Consensus protocol || **Failover** | ❌ None | ⚠️ Slow (heartbeat timeout) | ✅ Fast (ephemeral nodes) || **Split-brain safe** | ❌ | ⚠️ Possible in edge cases | ✅ Impossible (ZAB) || **Resource usage** | Low | High (constant polling) | Low (event-driven) || **Complexity** | None | Medium | Medium (needs ZK cluster) |### Key TakeawayZooKeeper's **sequential ephemeral nodes** make leader election reliable:- **Sequential**: Gives each candidate a unique, ordered number- **Ephemeral**: Automatically removed when the server disconnects- Together: The lowest-numbered surviving node is always the leader### When to Use ThisUse ZooKeeper leader election when:- You need exactly one leader at all times- You need automatic failover when the leader crashes- You can't tolerate split-brain (two leaders at once)Examples: database primary/replica management, task scheduler coordination, partition assignment

In [ ]:
# Cleanupif zk.exists("/demo"):    zk.delete("/demo", recursive=True)zk.stop()print("Cleaned up ZooKeeper nodes. Done!")